# Preprocessing GSE254569 Validation Cohort

This notebook splits the large GSE254569 AnnData object into donor-specific files and generates pseudobulk expression data for major cell types.
Metadata is downloaded from https://www.nature.com/articles/s41593-024-01742-z (supplementary table 2)

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import gc
import sys
import anndata as ad
from scipy.sparse import csr_matrix
import json
from pandas.api.types import is_object_dtype, is_categorical_dtype

# Add sources to path
sys.path.append('../../sources')
import pseudobulk_ct as pc

# Paths
root_path = "/data/home/swkim0523/research/_SCZ_mic_ast_fin_0511"
data_dir = f"{root_path}/data/validation_cohort/GSE254569"
raw_adata_path = f"{data_dir}/GSE254569_adata_RNA.h5ad"
metadata_path = f"{data_dir}/41593_2024_1742_MOESM2_ESM.xlsx"

donor_adata_dir = f"{data_dir}/adata_by_donor"
pb_ct_dir = f"{data_dir}/pb_ct"

os.makedirs(donor_adata_dir, exist_ok=True)
os.makedirs(pb_ct_dir, exist_ok=True)

/data/home/swkim0523/.local/lib/python3.9/site-packages/setuptools_scm/version.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import iter_entry_points


## Step 1: Split Large AnnData by Donor

The raw AnnData file is very large (~55GB). We split it by donor to make subsequent processing more manageable.

In [2]:
print("Loading large AnnData object...")
adata = sc.read_h5ad(raw_adata_path)
donors = adata.obs['Donor'].unique()
print(f"Found {len(donors)} donors.")

for donor in tqdm(donors, desc="Splitting donors"):
    save_path = f"{donor_adata_dir}/GSE254569_adata_{donor}.h5ad"
    if os.path.exists(save_path):
        continue
        
    adata_donor = adata[adata.obs['Donor'] == donor].copy()
    condition = adata_donor.obs['Classification'].unique()[0]
    # Rename file to include condition for easier tracking later
    final_save_path = f"{donor_adata_dir}/GSE254569_adata_{donor}_{condition}.h5ad"
    
    if not os.path.exists(final_save_path):
        adata_donor.write_h5ad(final_save_path)
    
    del adata_donor
    gc.collect()

del adata
gc.collect()
print("Splitting complete.")

Loading large AnnData object...
Found 87 donors.


Splitting donors: 100%|██████████| 87/87 [03:31<00:00,  2.43s/it]

Splitting complete.


## Step 2: Metadata and Common Genes

In [3]:
# Load Metadata
df_meta = pd.read_excel(metadata_path, sheet_name='Table S2')
df_meta.columns = df_meta.iloc[1]
df_meta = df_meta.drop(index=[0, 1])
df_meta = df_meta.reset_index(drop=True)

path_list = []
for idx, row in df_meta.iterrows():
    donor = row['Donor']
    condition = row['Classification']
    path = f"{donor_adata_dir}/GSE254569_adata_{donor}_{condition}.h5ad"
    if os.path.exists(path):
        path_list.append(path)

print(f"Found {len(path_list)} donor files.")

# Find Common Genes using pc.get_common_genes
gene_names = pc.get_common_genes(path_list)
print(f"Found {len(gene_names)} common genes.")

Found 87 donor files.


Scanning genes: 100%|██████████| 87/87 [01:39<00:00,  1.14s/it]

Found 26195 common genes.


## Step 3: Pseudobulk Generation

In [4]:
# Define cell types and column
ct_col = 'major_celltypes'
cell_types = ['Exc_Neurons', 'Astrocytes', 'Endothelial', 'OPC', 'Microglia', 'In_Neurons', 'Oligodendrocyte']

def get_pseudobulk_by_cell_type_val(gene_names, path_list, ct_col, cell_types):
    # Adjusted version of pc.get_pseudobulk_by_cell_type for this naming convention
    mean_bank = {ct: [] for ct in cell_types}
    donor_ids = []

    for p in tqdm(path_list, desc="Processing donors"):
        adata = sc.read_h5ad(p)
        adata = adata[:, gene_names]
        # Extract donor ID from path
        # GSE254569_adata_s1_Schizophrenia.h5ad -> s1
        donor_id = os.path.basename(p).split("_")[2]
        donor_ids.append(donor_id)

        for ct in cell_types:
            mask = adata.obs[ct_col] == ct
            if mask.sum() == 0:
                # If a donor is missing a cell type, we'll fill with zeros or handle as needed.
                # The defense code raised a ValueError, but we might want to check.
                mean_expr = np.zeros(len(gene_names))
            else:
                mean_expr = adata[mask].X.mean(axis=0)
                if not isinstance(mean_expr, np.ndarray):
                    mean_expr = np.asarray(mean_expr).ravel()
            mean_bank[ct].append(mean_expr)

        del adata
        gc.collect()

    pb_ct = {}
    for ct in cell_types:
        mat = np.stack(mean_bank[ct])
        adata_ct = ad.AnnData(
            X   = mat,
            obs = pd.DataFrame(index=donor_ids),
            var = pd.DataFrame(index=gene_names)
        )
        pb_ct[ct] = adata_ct
    return pb_ct

pb_ct = get_pseudobulk_by_cell_type_val(gene_names, path_list, ct_col, cell_types)

Processing donors: 100%|██████████| 87/87 [02:15<00:00,  1.56s/it]


## Step 4: Metadata Merging and Saving

In [5]:
def sanitize_obs(df):
    df = df.copy()
    for c in df.columns:
        s = df[c]
        if isinstance(s.dtype, pd.CategoricalDtype):
            df[c] = s.astype(str)
            continue
        if is_object_dtype(s):
            def conv(x):
                if isinstance(x, (list, dict, tuple, set, slice, range)):
                    return json.dumps(list(x) if isinstance(x, set) else x)
                if x is None:
                    return ""
                return str(x)
            df[c] = s.map(conv)
    df.index = df.index.astype(str)
    return df

for ct in pb_ct.keys():
    print(f"Finalizing {ct}...")
    adata_ct = pb_ct[ct]
    # Add Donor column to obs for joining
    adata_ct.obs['Donor'] = adata_ct.obs.index
    # Join metadata
    adata_ct.obs = adata_ct.obs.join(df_meta.set_index('Donor'), on='Donor')
    # Sanitize and Save
    adata_ct.obs = sanitize_obs(adata_ct.obs)
    adata_ct.X = csr_matrix(adata_ct.X)
    adata_ct.write_h5ad(f"{pb_ct_dir}/pb_{ct}_avg.h5ad")
    
print("All pseudobulk files saved in:", pb_ct_dir)

Finalizing Exc_Neurons...
Finalizing Astrocytes...
Finalizing Endothelial...
Finalizing OPC...
Finalizing Microglia...
Finalizing In_Neurons...
Finalizing Oligodendrocyte...
All pseudobulk files saved in: /data/home/swkim0523/research/_SCZ_mic_ast_fin_0511/data/validation_cohort/GSE254569/pb_ct
